# Naive baseline

For this, we assume we have access to training data sampled sparsely from the underling conditional temporal process (x, y, t). 

We then want to train a NN $\Phi(x, t) = y$, in a supervised manner. Then, we can sample from this naive surrogate at arbitrary t, with condition x, to get arbitrary temporal distribution

In [2]:
import time
import torch
from torch import nn, Tensor
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
from sklearn.datasets import make_moons
import numpy as np
from scipy.stats import vonmises as scipy_vonmises
from scipy.stats import lognorm as scipy_lognorm
warnings.filterwarnings("ignore", category=UserWarning, module='torch')

In [5]:
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

Using gpu


In [6]:
torch.manual_seed(42)

In [7]:
def semicircle_data():
    """
    Loads the semicircle dataset.
    """

    data = torch.load("../NLOT/data/conditional_semicircles.pt")
    data = data.to(device)
    return data

In [8]:
semicircle_data = semicircle_data()

In [22]:
#concat 0 to each entry in semicircle_data[0,:,:]

semicircle_data_time_0 = torch.cat((semicircle_data[0,:,:], torch.zeros(semicircle_data[0,:,:].shape[0], 1, device=device)), dim=1)
semicircle_data_time_05 = torch.cat((semicircle_data[1,:,:], torch.ones(semicircle_data[1,:,:].shape[0], 1, device=device) * 0.5), dim=1)
semicircle_data_time_1 = torch.cat((semicircle_data[2,:,:], torch.ones(semicircle_data[2,:,:].shape[0], 1, device=device)), dim=1)

flattened_semicircle_data = torch.cat((semicircle_data_time_0, semicircle_data_time_05, semicircle_data_time_1), dim=0)

In [27]:
output = flattened_semicircle_data[:, :2]
input = flattened_semicircle_data[:, 2:]

In [28]:
output

tensor([[-0.1920,  0.5221],
        [-0.0387,  0.1457],
        [-0.2248,  0.4138],
        ...,
        [ 1.8934,  0.1850],
        [ 2.0822,  0.1160],
        [ 1.8804,  0.2457]], device='cuda:0')

In [29]:
input

tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        ...,
        [3., 1.],
        [3., 1.],
        [3., 1.]], device='cuda:0')